# 09 · Dataset Preparation for Fine-Tuning

In plain English, this is the lesson where we **build the dataset** that the rest of the course is built on. A fine-tuned model is only as good as the data you feed it — give it messy, inconsistent, or unbalanced examples and it will learn messy, inconsistent, unbalanced habits. So before we touch any fancy training technique, we'll create a clean, well-organized dataset for a realistic business problem, save it in the standard **JSONL** format, and split it into **training** and **validation** sets.

Everything here is small and runs instantly on a plain laptop CPU — no GPU, no big downloads. The files you create in this notebook (`leads.jsonl`, `train.jsonl`, `val.jsonl`) are **reused by every later notebook (10–15)**, so this lesson is the foundation for all the fine-tuning ahead.

## What you'll learn

- What makes a **good** fine-tuning dataset: a **consistent format**, **clean labels**, **enough examples per class**, and **no leakage** between train and validation.
- The **lead-intent** business problem we'll use for the rest of the course, and how to **generate** its canonical dataset.
- How to **explore** the data with `pandas`: view rows, check **class balance** with `value_counts()`, and run basic **sanity checks**.
- The **JSONL** format ("one JSON object per line") — how to **write** the dataset to `leads.jsonl` and **read it back** with `json.loads`.
- How to make a seeded **train/validation split** (~80/20), why we hold out validation, and how to check the split is **balanced** (the idea behind *stratification*).
- Practical **data-cleaning** ideas: deduplication, handling **class imbalance**, and catching **label typos**.
- A short note on **`max_length`**: longer text means more tokens means more memory.

## Why this matters for fine-tuning

Fine-tuning is mostly a **data** problem, not a coding problem. The model architecture, the optimizer, the training loop — those are largely solved and handed to you by libraries like `transformers`. What you actually control, and what decides whether your fine-tune succeeds or fails, is the **dataset**.

A great dataset has four properties:

1. **Consistent format** — every example has the same fields in the same shape, so the training code never has to guess.
2. **Clean labels** — the target column uses a small, fixed set of correct values (no typos, no surprises).
3. **Enough examples per class** — the model needs to see each category enough times to learn it.
4. **No leakage** — the validation examples are *held back* and never seen during training, so your accuracy number is honest.

In this notebook we'll build a dataset that has all four, and save it to disk. Then notebooks 10–15 will load these exact files to demonstrate instruction formatting, full fine-tuning, LoRA, QLoRA, evaluation, and the capstone. Get the data right here and everything downstream gets easier.

## Setup

Run the cell below once. The `%pip install` line is **commented out** — uncomment it only if `pandas` isn't already installed (for example on a fresh environment or Google Colab).

We import three standard tools:
- **`json`** — to read and write JSON (the format inside each line of our `.jsonl` files).
- **`random`** — to generate the dataset and to shuffle it reproducibly.
- **`pandas as pd`** — a table/spreadsheet library that makes exploring data easy.

In [ ]:
# Uncomment the next line if pandas isn't installed (Colab / fresh environment):
# %pip install pandas

import json      # read/write JSON objects (one per line in JSONL files)
import random    # generate and shuffle data reproducibly
import pandas as pd  # tables for exploring the dataset

print("pandas version:", pd.__version__)
print("Setup complete — ready to build a dataset.")

## 1. What a good fine-tuning dataset looks like

Before generating anything, let's make the four properties concrete with a tiny example. Below are two versions of the *same* three records. One is messy; one is clean. Read both and notice the difference.

**Messy (don't do this):**

```text
{"size": 3, "Income": "70k",  "intent": "Hot"}
{"family_size": 3, "income": 70000, "lead_intent": "hot"}
{"family_size": 3, "income": 70000, "lead_intent": "hott"}   # typo!
```

Problems: the field names change (`size` vs `family_size`), the income is sometimes text (`"70k"`) and sometimes a number, the label is capitalized inconsistently (`"Hot"` vs `"hot"`), and one label is a **typo** (`"hott"`). Training code would choke or, worse, silently learn garbage.

**Clean (do this):**

```text
{"family_size": 3, "income": 70000, "lead_intent": "hot"}
{"family_size": 3, "income": 70000, "lead_intent": "warm"}
{"family_size": 3, "income": 70000, "lead_intent": "cold"}
```

Same fields, same types, labels drawn from a fixed set `{"hot", "warm", "cold"}`. That consistency is the whole game. The generator we write next produces clean data *by construction*, but later in this notebook we'll still run checks — because real-world data is never this tidy.

## 2. The lead-intent problem (and our canonical dataset)

**The business problem.** Imagine you work at a company that sells a product or service. People interact with your website — they request a quote, book a demo, download a brochure, or sign up for a newsletter. Each of these people is a **lead**. Your sales team can't call everyone, so they want to know: *how likely is this lead to buy?*

We'll frame this as a **classification** task. Each lead gets a label, **`lead_intent`**, which is one of three values:

- **`"hot"`** — very likely to buy, call them first.
- **`"warm"`** — interested, worth following up.
- **`"cold"`** — just looking, low priority.

The label depends on signals like family size, income, whether they rent or own, what action they took (their *CTA*, or "call to action"), and how urgent their need is.

The cell below **generates 200 leads** with a simple scoring rule. This exact generator is the **canonical dataset** for the whole course — every later notebook starts from the files it produces. We include it inline so this notebook is fully self-contained. The `random.seed(42)` line makes the data **identical every time you run it**, so your results match the rest of the course.

In [ ]:
import random
random.seed(42)
MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
CTAS = ["requested_quote","booked_demo","downloaded_brochure","newsletter_signup"]
CONDITIONS = ["urgent","exploring","just_browsing"]

def make_lead():
    family_size = random.randint(1, 6)
    income = random.choice([25000,40000,55000,70000,85000,100000,120000,150000])
    rent_or_own = random.choice(["rent","own"])
    cta = random.choice(CTAS)
    engagement_month = random.choice(MONTHS)
    current_condition = random.choice(CONDITIONS)
    score = 0
    if cta in ("requested_quote","booked_demo"): score += 2
    elif cta == "downloaded_brochure": score += 1
    if rent_or_own == "own": score += 1
    if income >= 80000: score += 1
    if family_size >= 4: score += 1
    if current_condition == "urgent": score += 2
    elif current_condition == "exploring": score += 1
    lead_intent = "hot" if score >= 5 else ("warm" if score >= 3 else "cold")
    return {"family_size": family_size, "income": income, "rent_or_own": rent_or_own,
            "cta": cta, "engagement_month": engagement_month,
            "current_condition": current_condition, "lead_intent": lead_intent}

leads = [make_lead() for _ in range(200)]

print("Generated", len(leads), "leads.")
print("First lead:", leads[0])
# Expected: a dict with family_size, income, rent_or_own, cta,
# engagement_month, current_condition, and a lead_intent of hot/warm/cold.

**What this does:**

- `random.seed(42)` fixes the random number generator so the 200 leads are **the same every run** — essential for a reproducible course.
- `make_lead()` builds **one** lead: it randomly picks each feature, computes a hidden **`score`** from simple rules (e.g. requesting a quote adds 2 points, being "urgent" adds 2), then converts that score into the label **`hot` / `warm` / `cold`**.
- `leads = [make_lead() for _ in range(200)]` is a **list comprehension** that calls `make_lead()` 200 times, giving us a list of 200 dictionaries — our raw dataset.

Notice the dataset is already **consistent** (same keys, same types) and the labels come from a **fixed set** — properties 1 and 2 from Section 1, for free.

### ✏️ Exercise

Change `range(200)` to `range(10)` in a **scratch copy** and print all ten leads to eyeball them. Then change it back to 200. (Tip: don't modify the canonical cell above — copy the line into the cell below so the rest of the notebook still uses the full 200.)

In [ ]:
# Your turn: make a small sample to inspect by eye.
# random.seed(42)
# sample = [make_lead() for _ in range(10)]
# for s in sample:
#     print(s)

## 3. Explore the data with pandas

A list of 200 dictionaries is hard to read. **`pandas`** turns it into a **DataFrame** — a table with named columns, exactly like a spreadsheet. From there we can view rows, count things, and run sanity checks in one line each.

`pd.DataFrame(leads)` is all it takes: pandas reads the dictionary keys as **column names** and each dict as a **row**.

In [ ]:
df = pd.DataFrame(leads)   # list of dicts -> table

print("shape (rows, columns):", df.shape)   # -> (200, 7)
print("column names:", list(df.columns))

# .head() shows the first 5 rows so you can see the structure:
df.head()

**What this does:**

- `pd.DataFrame(leads)` converts our 200 dicts into a table with 7 columns.
- `df.shape` reports `(rows, columns)` — here `(200, 7)`.
- `df.head()` displays the **first 5 rows**. In a notebook, returning a DataFrame on the last line renders it as a nice table.

### Class balance: `value_counts()`

The single most important thing to check in a classification dataset is **class balance** — how many examples you have of each label. If 95% of your leads are `"cold"`, a lazy model could "score" 95% just by always guessing `"cold"`, and never learn the others. `value_counts()` counts how often each label appears.

In [ ]:
# How many of each label? (the heart of "enough examples per class")
counts = df["lead_intent"].value_counts()
print("Counts per class:")
print(counts)

print("\nProportions per class:")
print(df["lead_intent"].value_counts(normalize=True).round(3))
# normalize=True turns counts into fractions that sum to 1.0.

**What this does:**

- `df["lead_intent"]` selects just the label column.
- `.value_counts()` counts each distinct value — e.g. how many `hot`, `warm`, and `cold` leads there are.
- `value_counts(normalize=True)` gives the same thing as **proportions** (fractions of the total), and `.round(3)` trims the decimals.

Our scoring rule makes `warm` the most common and `hot`/`cold` rarer — the classes are **somewhat imbalanced**, which is realistic. We'll talk about what to do about that in Section 7.

### Basic sanity checks

Three quick checks catch most data problems before they reach training:

1. Are there any **missing values** (`NaN`)?
2. Does each column hold the **types/values** we expect?
3. Are the labels exactly the **set we intended**?

In [ ]:
# 1) Missing values per column (we expect all zeros):
print("Missing values per column:")
print(df.isna().sum())

# 2) Quick numeric summary of the numeric columns:
print("\nNumeric summary:")
print(df[["family_size", "income"]].describe())

# 3) The exact set of labels actually present:
print("\nUnique labels present:", sorted(df["lead_intent"].unique()))
# Expected: ['cold', 'hot', 'warm']  -- exactly our three intended labels.

**What this does:**

- `df.isna().sum()` counts missing values in each column. All zeros means no holes in the data.
- `df[[...]].describe()` prints count/mean/min/max etc. for the numeric columns — a fast way to spot impossible values (a negative income would jump out here).
- `sorted(df["lead_intent"].unique())` lists every distinct label. We confirm it's exactly `['cold', 'hot', 'warm']` — no typos, no extra categories.

### ✏️ Exercise

Use `value_counts()` on a **different** column, like `df["cta"]` or `df["current_condition"]`, to see how those features are distributed. Then try `df.groupby("cta")["lead_intent"].value_counts()` to see how the label breaks down *within* each CTA — do "requested_quote" leads skew hotter?

In [ ]:
# Your turn:
# print(df["cta"].value_counts())
# print(df.groupby("cta")["lead_intent"].value_counts())

## 4. The JSONL format: one JSON object per line

Fine-tuning tools almost universally expect data as **JSONL** — short for "**JSON Lines**." The rule is dead simple:

> **One JSON object per line.** Each line is a complete, valid JSON record; the lines together make the dataset.

A `.jsonl` file for three leads looks like this (each line is independent):

```text
{"family_size": 3, "income": 70000, "lead_intent": "warm"}
{"family_size": 5, "income": 120000, "lead_intent": "hot"}
{"family_size": 1, "income": 25000, "lead_intent": "cold"}
```

Why JSONL instead of one big JSON array? Because you can read it **one line at a time** without loading the whole file into memory — which matters when datasets get huge — and you can append a new record just by adding a line. Let's write our 200 leads to `leads.jsonl`.

In [ ]:
# Write the dataset to JSONL: one json.dumps(...) per line.
with open("leads.jsonl", "w", encoding="utf-8") as f:
    for lead in leads:
        line = json.dumps(lead)   # turn one dict into a JSON string
        f.write(line + "\n")      # write it, then a newline to end the line

print("Wrote", len(leads), "leads to leads.jsonl")

**What this does:**

- `open("leads.jsonl", "w")` opens the file for **writing** (`"w"` overwrites any existing file).
- For each lead, `json.dumps(lead)` converts the Python dict into a **JSON string** (e.g. `{"family_size": 3, ...}`).
- `f.write(line + "\n")` writes that string followed by a **newline** (`\n`), so each record sits on its own line — the JSONL rule.

### Read it back

Reading JSONL is the mirror image: loop over the file's lines, and for each line use `json.loads(...)` to turn the string back into a Python dict.

In [ ]:
# Read the JSONL back into a list of dicts.
loaded = []
with open("leads.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()       # remove the trailing newline/whitespace
        if not line:              # skip any accidental blank lines
            continue
        loaded.append(json.loads(line))   # JSON string -> Python dict

print("Read back", len(loaded), "leads.")
print("First loaded lead:", loaded[0])
print("Matches original:", loaded[0] == leads[0])  # -> True

**What this does:**

- We loop over the file line by line (`for line in f`), `strip()` the newline so `json.loads` sees clean text, and skip empty lines just to be safe.
- `json.loads(line)` parses each JSON string **back into a dict**; we collect them in `loaded`.
- The check `loaded[0] == leads[0]` prints `True`, proving the round-trip (write then read) preserved the data exactly.

Below, we also print the **first two raw lines** of the file verbatim — that's what "one JSON object per line" really looks like on disk.

In [ ]:
# Show the first two raw lines of the file, exactly as stored.
with open("leads.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(f"line {i}: {line.rstrip()}")   # rstrip drops the trailing newline
        if i == 1:
            break
# Each printed line is a full JSON object — that's the whole format.

### ✏️ Exercise

Write a tiny helper `load_jsonl(path)` that opens a file, loops its lines, and returns a list of dicts (the read-back logic above). Then call it on `"leads.jsonl"` and confirm `len(...)` is 200. You'll reuse this helper in later notebooks.

In [ ]:
# Your turn:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

# print(len(load_jsonl("leads.jsonl")))   # expect 200

## 5. Train / validation split

We never train on **all** our data. We hold a chunk back as a **validation set** — examples the model **does not see** during training — so we can measure how well it does on data it hasn't memorized. This is property #4 from Section 1: **no leakage**.

Think of it like studying for an exam. If you practice on the exact questions that will be on the test, a high score proves nothing — you just memorized. The validation set is a *fresh* set of questions that tells you whether you actually learned the pattern.

The plan:
1. **Shuffle** the data (with a seed, so it's reproducible).
2. Take the **first ~80%** as `train`, the **last ~20%** as `val`.
3. Save each to its own JSONL file.

In [ ]:
# 1) Shuffle a COPY so we don't disturb the original `leads` list.
shuffled = leads[:]            # [:] makes a shallow copy of the list
random.seed(123)               # a fixed seed -> the same shuffle every run
random.shuffle(shuffled)       # shuffle in place

# 2) Compute the split point at 80%.
split_idx = int(0.8 * len(shuffled))   # 0.8 * 200 = 160
train_rows = shuffled[:split_idx]      # first 160
val_rows   = shuffled[split_idx:]      # last 40

print("total:", len(shuffled))
print("train:", len(train_rows))   # -> 160
print("val:  ", len(val_rows))     # -> 40

**What this does:**

- `leads[:]` makes a **copy** so shuffling doesn't scramble the original list (which the JSONL writing above relied on).
- `random.seed(123)` before `random.shuffle(...)` makes the shuffle **reproducible** — same order every run, so your train/val split matches everyone else's.
- `split_idx = int(0.8 * len(shuffled))` is the cut point: 80% of 200 = 160. We slice the first 160 rows into `train` and the remaining 40 into `val`.

**Why shuffle first?** If the data had any hidden order (say, all the `hot` leads were generated last), an unshuffled split could put them *all* in validation. Shuffling spreads every class across both halves.

### Check the split is balanced (the idea of *stratification*)

After splitting, we want each class to appear in **roughly the same proportions** in train and val. If `hot` is 20% of the full data, it should be about 20% of train *and* about 20% of val. When you deliberately enforce that, it's called **stratified** splitting. Our random shuffle usually gets close to it for free — let's verify.

In [ ]:
def label_proportions(rows):
    """Return each label's share of the given list of lead dicts."""
    s = pd.Series([r["lead_intent"] for r in rows])
    return s.value_counts(normalize=True).round(3)

print("FULL proportions:")
print(label_proportions(leads))
print("\nTRAIN proportions:")
print(label_proportions(train_rows))
print("\nVAL proportions:")
print(label_proportions(val_rows))
# The three should look SIMILAR. Small wobbles are normal with only 40 val rows.

**What this does:**

- `label_proportions` pulls the `lead_intent` from each row into a pandas `Series`, then uses `value_counts(normalize=True)` to get each label's **fraction**.
- We print the proportions for the **full** data, **train**, and **val**.
- They should be **close to each other**. They won't match perfectly — with only 40 validation rows, each row is 2.5%, so some wobble is expected. If train and val looked *wildly* different, that would be a red flag worth fixing (e.g. by stratifying explicitly).

### Save `train.jsonl` and `val.jsonl`

Now write both splits to disk in JSONL. We'll factor the writing into a small reusable function — you'll use these two files in every notebook from here on.

In [ ]:
def write_jsonl(path, rows):
    """Write a list of dicts to `path`, one JSON object per line."""
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")
    print(f"Wrote {len(rows):>3} rows to {path}")

write_jsonl("train.jsonl", train_rows)
write_jsonl("val.jsonl",   val_rows)

# Sanity check: read them back and count.
print("train.jsonl has", len(load_jsonl("train.jsonl")), "rows")
print("val.jsonl   has", len(load_jsonl("val.jsonl")),   "rows")

**What this does:**

- `write_jsonl` is the write loop from Section 4, wrapped so we can call it twice without repeating ourselves.
- We write `train_rows` to `train.jsonl` and `val_rows` to `val.jsonl`.
- The final two lines reuse the `load_jsonl` helper to read both files back and confirm the counts (160 and 40). These three files — `leads.jsonl`, `train.jsonl`, `val.jsonl` — are now your **course dataset**.

### ✏️ Exercise

Change the split to **90/10** by setting the fraction to `0.9`, re-run the split and save cells, and confirm you get 180 train / 20 val. Then think about the trade-off: a bigger train set gives the model more to learn from, but a tiny val set (20 rows) gives a **noisier** accuracy estimate. (After experimenting, switch back to 0.8 so your files match the rest of the course.)

In [ ]:
# Your turn: copy the split cell and change 0.8 to 0.9, then re-check the sizes.

## 6. Data cleaning: duplicates and label typos

Our generated data is clean, but **real** data never is. Two of the most common problems are **duplicate rows** and **label typos**. Let's deliberately create a small messy dataset and clean it, so you know the moves.

### Deduplication

Duplicate examples waste training time and can quietly **bias** the model toward whatever the repeated rows say. They're also a sneaky source of **leakage**: if the *same* row lands in both train and val, your validation score is inflated.

In [ ]:
# A tiny messy dataset with an exact duplicate.
messy = [
    {"family_size": 2, "income": 40000, "lead_intent": "cold"},
    {"family_size": 5, "income": 120000, "lead_intent": "hot"},
    {"family_size": 2, "income": 40000, "lead_intent": "cold"},  # exact duplicate
    {"family_size": 3, "income": 70000, "lead_intent": "warm"},
]

messy_df = pd.DataFrame(messy)
print("before dedup:", len(messy_df), "rows")

# drop_duplicates() removes rows that are identical across all columns.
clean_df = messy_df.drop_duplicates().reset_index(drop=True)
print("after  dedup:", len(clean_df), "rows")   # -> 3
clean_df

**What this does:**

- We build a 4-row table where row 0 and row 2 are **identical**.
- `messy_df.drop_duplicates()` removes the repeat, leaving 3 unique rows.
- `.reset_index(drop=True)` renumbers the rows `0,1,2` (otherwise the old index `0,1,3` would remain). The duplicate is gone.

### Catching label typos

Property #2 was *clean labels*. The fastest way to catch a stray label like `"hott"` or `"Warm"` is to compare the labels actually present against the **allowed set** you defined up front.

In [ ]:
ALLOWED_LABELS = {"hot", "warm", "cold"}   # the only labels we permit

# A dataset with a sneaky typo and a wrong-case label.
typo_rows = [
    {"family_size": 2, "income": 40000, "lead_intent": "cold"},
    {"family_size": 5, "income": 120000, "lead_intent": "hott"},   # typo
    {"family_size": 3, "income": 70000, "lead_intent": "Warm"},    # wrong case
]

# Find any rows whose label isn't in the allowed set.
bad = [r for r in typo_rows if r["lead_intent"] not in ALLOWED_LABELS]
print("Bad labels found:", [r["lead_intent"] for r in bad])
# -> ['hott', 'Warm']

# Confirm our REAL dataset is clean against the same check:
real_bad = [r for r in leads if r["lead_intent"] not in ALLOWED_LABELS]
print("Bad labels in the real dataset:", len(real_bad))   # -> 0

**What this does:**

- `ALLOWED_LABELS` is the **fixed set** of valid labels — your source of truth.
- The list comprehension keeps only rows whose label is **not** in that set, flagging `"hott"` (typo) and `"Warm"` (wrong case).
- Running the same check on our real `leads` returns **0** bad rows — confirming the canonical dataset is clean. In a real project you'd run this check every time you load data.

### ✏️ Exercise

Write a one-line check that also catches a **missing** label — a row where `lead_intent` is an empty string `""` or the key is absent. Hint: use `r.get("lead_intent")` so a missing key returns `None` instead of raising an error, and treat `None`/`""` as bad.

In [ ]:
# Your turn:
# rows_to_check = typo_rows + [{"family_size": 1, "income": 25000}]  # last row has NO label
# bad2 = [r for r in rows_to_check
#         if r.get("lead_intent") not in ALLOWED_LABELS]
# print([r.get("lead_intent") for r in bad2])

## 7. Handling class imbalance

Back in Section 3 we saw the classes aren't equal — `warm` is common, `hot`/`cold` are rarer. That's **class imbalance**, and it's property #3 from Section 1 ("enough examples per class") in action. When one class is much rarer, the model sees it less and tends to under-predict it.

You don't always need to fix imbalance, but it's good to know your options:

- **Collect more data** for the rare class (best, if you can).
- **Oversample** the rare class — duplicate (or slightly vary) its examples so it shows up more often during training.
- **Undersample** the common class — drop some of its examples so the counts even out (you lose data, though).
- **Class weights** — tell the loss function to "care more" about mistakes on the rare class (a training-side fix, covered later).

Below is a simple **oversampling** demo: we duplicate the rarest class until it matches the most common one. We do this on a *copy* — we are **not** changing our saved files, just showing the technique.

In [ ]:
# Group the rows by label.
by_label = {}
for r in leads:
    by_label.setdefault(r["lead_intent"], []).append(r)

for label, rows in by_label.items():
    print(f"{label:>5}: {len(rows)} rows")

# Find the size of the biggest class -- our oversampling target.
target = max(len(rows) for rows in by_label.values())
print("\nTarget size per class (oversample up to):", target)

random.seed(7)
oversampled = []
for label, rows in by_label.items():
    rows = rows[:]                          # copy this class's rows
    while len(rows) < target:               # repeat until it reaches target
        rows.append(random.choice(rows))    # add a random existing example
    oversampled.extend(rows)

# Confirm every class now has the same count.
bal = pd.Series([r["lead_intent"] for r in oversampled]).value_counts()
print("After oversampling:")
print(bal)
print("\nTotal rows now:", len(oversampled))

**What this does:**

- We build `by_label`, a dict mapping each label to the list of its rows (`setdefault` creates an empty list the first time it sees a label), and find `target` — the size of the **largest** class.
- For each class we **append random copies of its own rows** until it reaches `target`, then collect everything in `oversampled`. The final `value_counts()` shows all three classes are now equal.

**Important caveat:** oversampling duplicates rows, so you must do it **only on the training set, after the train/val split** — never before, or copies of the same row could land in both train and val (leakage!). Here we just demonstrated the mechanics on the full set.

### ✏️ Exercise

Modify the oversampling loop to bring every class up to `target * 2` instead of `target` (over-oversampling). Print the new counts. Then ask yourself: at some point, does adding *more* copies of the *same* rare examples actually teach the model anything new? (Answer: not really — which is why collecting genuinely new data beats oversampling when you can.)

In [ ]:
# Your turn: copy the oversampling loop and change `target` to `target * 2`.

## 8. A note on `max_length` (for later)

When we fine-tune, the model doesn't read raw dicts — it reads **text**, which gets split into **tokens** (you saw this in notebook 07). A setting called **`max_length`** caps how many tokens each example can be. It matters because:

> **Longer text → more tokens → more memory and more compute.**

Every example in a training batch is padded to the same length, so one very long example can blow up the memory for the whole batch. The trade-off:

- **Too small** a `max_length` → long examples get **truncated** (cut off), losing information.
- **Too large** → you waste memory and time padding short examples.

You pick `max_length` by looking at how long your examples actually are. Let's preview that by turning each lead into a short text sentence and measuring its length in **words** (a rough stand-in for tokens — real tokenization comes in notebook 10).

In [ ]:
def lead_to_text(lead):
    """Turn one lead dict into a short natural-language sentence."""
    return (f"A lead with family size {lead['family_size']}, "
            f"income {lead['income']}, who {lead['rent_or_own']}s their home, "
            f"took action '{lead['cta']}' in {lead['engagement_month']} "
            f"and is currently {lead['current_condition']}.")

example_text = lead_to_text(leads[0])
print("Example text:\n", example_text)

# Rough length in words for ALL leads (a proxy for token count).
word_lengths = [len(lead_to_text(l).split()) for l in leads]
print("\nShortest (words):", min(word_lengths))
print("Longest  (words):", max(word_lengths))
print("Average  (words):", round(sum(word_lengths) / len(word_lengths), 1))
# Tokens are usually a bit MORE than words, so pick max_length with headroom.

**What this does:**

- `lead_to_text` formats a lead as a readable sentence — this is a preview of the "instruction formatting" you'll do for real in notebook 10.
- We measure each sentence's length in **words** (`.split()` splits on spaces) as a quick proxy for token count.
- Knowing the shortest/longest/average length tells you a sensible `max_length`: big enough to fit the longest example without truncation, but not wastefully large. Tokens usually run a bit higher than word counts, so leave some headroom.

### ✏️ Exercise

Print the **full text** of the 5 longest leads (sort by word length and look at the top 5). Do they share anything — a longer CTA name, a particular month? This is the kind of quick look that helps you choose `max_length` confidently.

In [ ]:
# Your turn:
# pairs = sorted(((len(lead_to_text(l).split()), lead_to_text(l)) for l in leads),
#                reverse=True)
# for n, text in pairs[:5]:
#     print(n, "words:", text)

## Common mistakes & how to debug them

- **Leakage between train and val.** The #1 sin. If the same (or near-identical) row appears in both, your validation score lies. Always **deduplicate before splitting**, and do any **oversampling after the split**, on the train set only.
- **Splitting without shuffling.** If the data has hidden order, an unshuffled split can pile one class into val. Always **shuffle with a seed** first.
- **Forgetting the seed.** No `random.seed(...)` means a different dataset/split every run, and your results won't match the course. Seed the generator *and* the shuffle.
- **Inconsistent labels.** `"hot"`, `"Hot"`, and `"hott"` are three different labels to the model. Check every label against an **allowed set** on load.
- **Ignoring class balance.** If you never run `value_counts()`, you won't notice a class is starved — and a model that always guesses the majority class can look "accurate" while being useless.
- **Bad JSONL.** A common error is writing a real array (`[{...}, {...}]`) instead of one object per line, or forgetting the `"\n"` so two records share a line. Read the file back with `json.loads` per line to verify it parses.
- **`JSONDecodeError` when reading.** Usually a blank line or a half-written file. Strip each line and skip empties (as our `load_jsonl` does), and make sure the writer finished.
- **`max_length` too small.** If important info gets cut off, the model can't learn from it. Measure your example lengths before choosing.

## Summary

- A good fine-tuning dataset has four properties: **consistent format**, **clean labels**, **enough examples per class**, and **no leakage** between train and validation.
- We introduced the **lead-intent** business problem and generated the **canonical 200-lead dataset** (seeded, so it's identical every run) — the dataset every later notebook reuses.
- We explored it with **pandas**: `pd.DataFrame`, `.head()`, `value_counts()` for **class balance**, and sanity checks (`isna`, `describe`, `unique`).
- We learned **JSONL** — one JSON object per line — and **wrote** `leads.jsonl` then **read it back** with `json.loads`, peeking at the raw lines.
- We made a seeded **80/20 train/validation split**, saved `train.jsonl` and `val.jsonl`, and confirmed the class proportions stayed **similar** (the idea behind **stratification**).
- We practiced **data cleaning**: `drop_duplicates`, checking labels against an **allowed set**, and **oversampling** the rare class (only ever on the train set!).
- We previewed **`max_length`**: longer text means more tokens means more memory — measure your examples to choose it well.

You now have `leads.jsonl`, `train.jsonl`, and `val.jsonl` on disk. These are the raw materials for everything that follows.

## What to learn next

Next up: **`10_instruction_finetuning_format.ipynb`**. You have clean data and a train/val split — now you'll learn how to **format** each example the way an instruction-tuned LLM expects: turning a lead's fields into a **prompt** (the instruction/input) and a **response** (the target label), the exact shape used to fine-tune chat-style models. We'll build directly on the `train.jsonl` and `val.jsonl` files you just created.